# Mirage19 Data Exploration

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from collections import Counter
import time
import copy
import multiprocessing
import os

# Configuration
RANDOM_SEED = 2025

## Load Data
Loading the preprocessed pickle file containing biflows and labels.

In [ ]:
DATA_PATH = "../dataset/mirage/2019/mirage2019_LOPEZ_lopez_lopez_100P_4F_APP_xST_PAD_metadata.pickle"

print("Loading dataset...")
with open(DATA_PATH, "rb") as f:
    X_raw = np.array(pickle.load(f), dtype=np.float32) # Convert to numpy array
    y_raw = np.array(pickle.load(f)) # Convert to numpy array

print(f"Data shape: {X_raw.shape}")
print(f"Labels shape: {len(y_raw)}")

# Example sample
example_sample = 7000
print("\nExample Sample (First 5 packets):\n", X_raw[example_sample][:5])
print("Example Label:", y_raw[example_sample])


## Data exploration
Checking the mean number of packets per flow. If dst is -1, it is a padding packet.

In [ ]:
def count_packets(flow):
    """Calcola il numero di pacchetti validi in un singolo flusso."""
    # Usa la logica ottimizzata: trova l'indice del primo -1, altrimenti usa la lunghezza
    return next((i for i, packet in enumerate(flow) if packet[0] == -1), len(flow))

# Esempio
print("Example sample packet len: ", count_packets(X_raw[example_sample]))

In [ ]:
X_n_packets = []

X_dir_trimmed = []
X_payloadLen_trimmed = []
X_tcpWin_trimmed = []
X_iat_trimmed = []

for i, flow in enumerate(X_raw):
    num_packets = count_packets(flow)
    X_n_packets.append(num_packets)

    X_dir_trimmed.append(flow[:num_packets, 0])
    X_payloadLen_trimmed.append(flow[:num_packets, 1].sum()) # Questi ultimi 3 non hanno molto senso
    X_tcpWin_trimmed.append(flow[:num_packets, 2].mean())
    X_iat_trimmed.append(flow[:num_packets, 3].mean())

X_n_packets = np.array(X_n_packets)

print("Example sample packet length:", X_n_packets[example_sample])

print("Example sample direction:", X_dir_trimmed[example_sample])
print("Example sample payload length:", X_payloadLen_trimmed[example_sample])
print("Example sample TCP window:", X_tcpWin_trimmed[example_sample])
print("Example sample IAT:", X_iat_trimmed[example_sample])


In [ ]:
print("Mean packets per flow: ", np.mean(X_n_packets))
print("Standard deviation packets per flow: ", np.std(X_n_packets))

Analyzing class distribution and feature statistics.

In [ ]:
# Class Distribution
counts = Counter(y_raw)
df_counts = pd.DataFrame.from_dict(counts, orient='index', columns=['count']).sort_values('count', ascending=False)

plt.figure(figsize=(15, 6))
sns.barplot(x=df_counts.index, y=df_counts['count'], palette='viridis')
plt.xticks(rotation=90)
plt.xlabel("App")
plt.ylabel("Number of Samples")
plt.title("Class Distribution", fontsize=16)
plt.tight_layout()
plt.show()

Checking, for each class, the mean number of packets per flow.

In [ ]:
# Mean packet pre flow
data = {
    'y': y_raw,
    'x': X_n_packets,
    # 'dir': X_dir,
    'payloadLen': X_payloadLen_trimmed,
    'tcpWin': X_tcpWin_trimmed,
    'iat': X_iat_trimmed,
}
df = pd.DataFrame(data)

df_media = df.groupby('y')['x'].mean().reset_index()
df_media.columns = ['Etichetta', 'Media_X']
df_media = df_media.sort_values('Media_X', ascending=False)

plt.figure(figsize=(15, 6))

sns.barplot(
    x='Etichetta', 
    y='Media_X', 
    data=df_media, 
    palette='viridis'
)

plt.xticks(rotation=90)
plt.title("Mean packet Distribution", fontsize=16)
plt.xlabel("App")
plt.ylabel("Numero medio di pacchetti")
plt.tight_layout()
plt.show()

Standard Deviation for each class

In [ ]:
df_std = df.groupby('y')['x'].std().reset_index()

# Rinomina le colonne
df_std.columns = ['Etichetta', 'Deviazione_Standard_X']

# Ordina in base alla Deviazione Standard in modo decrescente
df_std = df_std.sort_values('Deviazione_Standard_X', ascending=False)


# --- 3. Visualizzazione con Seaborn ---
plt.figure(figsize=(15, 6))

# Usa sns.barplot con il DataFrame df_std ordinato
sns.barplot(
    x='Etichetta', 
    y='Deviazione_Standard_X', 
    data=df_std, 
    palette='magma'
)

# Impostazioni finali del grafico
plt.xticks(rotation=90)
plt.title("Standard Deviation", fontsize=16)
plt.xlabel("App")
plt.ylabel(" ")
plt.tight_layout()
plt.show()

In [ ]:
df_analisi_completa = df.groupby('y')['x'].agg(
    [
        'mean',       # Media
        'std',        # Deviazione Standard
        'median',     # Mediana
        'min',        # Minimo
        'max',        # Massimo
        pd.Series.skew, # Asimmetria (Skewness)
        pd.Series.kurt  # Curtosi (Kurtosis)
    ]
).reset_index()

print(df_analisi_completa)

## Preprocessing

In [ ]:
from sklearn.preprocessing import MinMaxScaler

N_PACKETS = 10
N_FEATURES = 4

def log1pPreproc(X):
    """
    Handles padding and normalizes features.
    X shape: (N, 36, 4)
    Features: [DIR, PL, TCPWIN, IAT]
    """
    X_proc = X.copy().astype(np.float32)

    # Mask for padding (using PL at index 1)
    is_padding = (X_proc[:, :, 1] == -1)

    # Set all features to 0 where is_padding is True
    mask = np.repeat(is_padding[:, :, np.newaxis], 4, axis=2)
    X_proc[mask] = 0

    # Apply Log1p to PL (1), TCPWIN (2), IAT (3)
    X_proc[:, :, 1:] = np.log1p(np.maximum(X_proc[:, :, 1:], 0))

    return X_proc[:, :N_PACKETS, :]


def minmaxPreproc(X):
    X_proc = X.copy().astype(np.float32)
    # min max scaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaler.fit(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = scaler.transform(np.reshape(X_proc, [-1, N_FEATURES]))
    X_proc = np.reshape(X_proc, [-1, X.shape[1], N_FEATURES])
    return X_proc[:, :N_PACKETS, :]


X_log = log1pPreproc(X_raw)
X_minmax = minmaxPreproc(X_raw)

# for i in range(20):
# numero random da 0 a len(X_raw)-1
random_sample = np.random.randint(0, len(X_raw))

for random_sample in [52180, 44261]:
    print("Random sample index:", random_sample, "App label:", y_raw[random_sample])

    # pl list
    pl = X_raw[random_sample, :, 1]
    max = np.max(pl[pl != -1])
    min = np.min(pl[pl != -1])
    print("Original PL - min:", min, "max:", max, "delta:", max - min)

    # log1p pl
    pl_log = X_log[random_sample, :, 1]
    max_log = np.max(pl_log)
    min_log = np.min(pl_log[pl_log != 0])
    print("Log1p PL - min:", min_log, "max:", max_log, "delta:", max_log - min_log)

    # minmax pl
    pl_minmax = X_minmax[random_sample, :, 1]
    max_minmax = np.max(pl_minmax)
    min_minmax = np.min(pl_minmax[pl_minmax != 0])
    print("MinMax PL - min:", min_minmax, "max:", max_minmax, "delta:", max_minmax - min_minmax)

    # Plot Packet Length of random sample before and after preprocessing
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(X_raw[random_sample, :N_PACKETS, 1], marker='o')
    plt.title("Original Packet Length")
    plt.xlabel("Packet Index")
    plt.ylabel("Packet Length")
    plt.ylim(-10, np.max(X_raw[:, :, 1]) * 1.1)
    plt.subplot(1, 3, 2)
    plt.plot(X_log[random_sample, :, 1], marker='o', color='orange')
    plt.title("Log1p Preprocessed Packet Length")
    plt.xlabel("Packet Index")
    plt.ylim(-0.1, np.max(X_log[:, :, 1]) * 1.1)
    plt.subplot(1, 3, 3)
    plt.plot(X_minmax[random_sample, :, 1], marker='o', color='green')
    plt.title("Min-Max Preprocessed Packet Length")
    plt.xlabel("Packet Index")
    plt.ylim(-0.1, 1.1)
    plt.tight_layout()
    plt.show()

# 52180 44261

# stessa cosa ma su dir feature
for random_sample in [52180, 44261]:
    print("Random sample index:", random_sample, "App label:", y_raw[random_sample])
    # Plot Direction of random sample before and after preprocessing
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1)
    plt.plot(X_raw[random_sample, :N_PACKETS, 0], marker='o')
    plt.title("Original Direction")
    plt.xlabel("Packet Index")
    plt.ylabel("Direction")
    plt.ylim(-1.1, 1.1)
    plt.subplot(1, 3, 2)
    plt.plot(X_log[random_sample, :, 0], marker='o', color='orange')
    plt.title("Log1p Preprocessed Direction")
    plt.xlabel("Packet Index")
    plt.ylim(-1.1, 1.1)
    plt.subplot(1, 3, 3)
    plt.plot(X_minmax[random_sample, :, 0], marker='o', color='green')
    plt.title("Min-Max Preprocessed Direction")
    plt.xlabel("Packet Index")
    plt.ylim(-1.1, 1.1)
    plt.tight_layout()
    plt.show()